# === USER CONFIGURATION ===

In [1]:
# # === USER CONFIGURATION ===
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ CAREFUL, please read the README_Notebooks.md file before
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

WORKDIR = "/home/pcastillo/sen2vm/WORKDIR"  # Working directory for sen2vm processing

# Path to downloaded product
PATH_L1B_DATA = "/home/pcastillo/sen2vm/WORKDIR/S2B_MSIL1B_20241019T120219_N0511_R023_20241022T154709.SAFE"

# Path to GIPP directory
PATH_GIPP = "/home/pcastillo/sen2vm/WORKDIR/GIPP"
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ If you do not have your own GIPP folder, you may use the inputs-download-notebook to download it, then you may indicate the path were you 
# /!\/!\/!\ downloaded it here in PATH_GIPP
# /!\/!\/!\ If you have your own GIPP folder, please note that this current notebook will search for a subfolder with mission S2[A/B/C] inside the GIPP folder
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

# Path to DEM directory 
PATH_DEM = "/home/pcastillo/sen2vm/WORKDIR/DEM"

# Path to GEOID directory
PATH_GEOID = "/home/pcastillo/sen2vm/WORKDIR/GEOID"
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ If you do not have your own GEOID folder, you may use the inputs-download-notebook to download it, then you may indicate the path were you
# /!\/!\/!\ downloaded it here in  
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

# Path to IERS file
PATH_IERS = "/home/pcastillo/sen2vm/WORKDIR"
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ If you do not have your own IERS, you may let this path empy (i.e. "") and execute the cell number 2 named #IERS Download
# /!\/!\/!\ In this case, the IERS will be downloaded in the WORKDIR
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

# The output folder is put in the WORKDIR
OUTPUT_FOLDER = WORKDIR + "/output/INVERSE"

# === SEN2VM OPTIONS ===
UTM_EPSG = 32628 # UTM zone EPSG code for the ROI
LOCATION = {
    "ul_x": 283910,
    "ul_y": 3641660,
    "lr_x": 354280,
    "lr_y": 3608316.0
}

# Grid step in pixels for sen2vm grid generation 
# These values represent the grid step in pixels, it's the pixel in the detector geometry, typically DEM resolution
# For S2 with DEM90, this gives approximately: 45m
STEPS = {
    "10m_bands": 45,   
    "20m_bands": 45,  
    "60m_bands": 45
}

# === GDAL ORTHO OPTIONS ===

ORTHO_SETTINGS = {
    "keep_bands": ["B02"],  # list of bands to keep for orthorectification
    "keep_detectors": ["01","02","03","04","05","06","07","08","09","10","11","12"]  # list of the detectors to keep 
}

# === Docker Options ===
# By setting REMOVE_DOCKER_IMAGE = False, the images are kept,
# which significantly speeds up subsequent executions.
#
# However, Docker images use disk space (~4 GB in this case).
#
# Docker images can be manually removed using:
#   docker images
#   docker rmi <image_id>
#
REMOVE_DOCKER_IMAGE = False  # True = remove Docker images after execution

# IERS Download

In [2]:
# === DOWNLOAD IERS ===
import os
import re
import requests
import sys
from datetime import datetime

if len(PATH_IERS) != 0: #Safety in case the PATH_IERS isn't empty
    print("Stopping because PATH_IERS is not empty")
    sys.exit(0)

if not os.path.exists(WORKDIR):
    raise RuntimeError(f"WORKDIR directory not found: {WORKDIR}")

print("Bulletin output directory:", WORKDIR)

PATH_IERS = WORKDIR # then our file will be there
# =====================================================================
# REMOVE EXISTING IERS BULLETINS
# =====================================================================

for f in os.listdir(WORKDIR):
    if f.startswith("bulletina-") and f.endswith(".txt"):
        os.remove(os.path.join(WORKDIR, f))
        print("Removed old bulletin A:", f)

    if f.startswith("bulletinb-") and f.endswith(".txt"):
        os.remove(os.path.join(WORKDIR, f))
        print("Removed old bulletin B:", f)

print("Cleanup of old bulletins complete.\n")
# =====================================================================
# EXTRACT PRODUCT DATE (FROM DATASTRIP)
# =====================================================================

datastrip_dir = os.path.join(PATH_L1B_DATA, "DATASTRIP")

if not os.path.isdir(datastrip_dir):
    raise RuntimeError(f"DATASTRIP directory not found: {datastrip_dir}")

datastrip_entries = os.listdir(datastrip_dir)
if not datastrip_entries:
    raise RuntimeError(f"No DATASTRIP found in: {datastrip_dir}")

# Take the first DATASTRIP product
datastrip_name = datastrip_entries[0]

match = re.search(r"_S(\d{8})T\d{6}_", datastrip_name)
if not match:
    raise RuntimeError("Could not extract product date from DATASTRIP name.")

product_date_str = match.group(1)

year = int(product_date_str[:4])
month = int(product_date_str[4:6])
day = int(product_date_str[6:8])

product_date = datetime(year, month, day)

print("Product date extracted from DATASTRIP:", product_date.date(), "\n")
# =====================================================================
# Roman conversion 
# =====================================================================

def int_to_roman(n):
    vals = [
        (1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'),
        (100, 'c'), (90, 'xc'), (50, 'l'), (40, 'xl'),
        (10, 'x'), (9, 'ix'), (5, 'v'), (4, 'iv'), (1, 'i')
    ]
    res = ""
    for v, s in vals:
        while n >= v:
            res += s
            n -= v
    return res
# =====================================================================
# Bulletin A
# =====================================================================

roman_year = int_to_roman(year - 1987)
doy = product_date.timetuple().tm_yday
index = (doy - 1) // 7 + 1

print(f"Bulletin A Roman year: {roman_year}")
print(f"Initial weekly index: {index}\n")

found = False

while index > 0:
    index_str = f"{index:03d}"
    url = f"https://datacenter.iers.org/data/6/bulletina-{roman_year}-{index_str}.txt"
    print("Trying bulletin:", url)

    response = requests.get(url)

    if response.status_code == 200:
        print("Bulletin found:", index_str)
        dest_file = os.path.join(WORKDIR, f"bulletina-{roman_year}-{index_str}.txt")
        found = True
        break

    index -= 1

if not found:
    raise RuntimeError("No Bulletin A available for current or previous weeks.")

with open(dest_file, "wb") as f:
    f.write(response.content)

print("Downloaded:", dest_file)

Stopping because PATH_IERS is not empty


SystemExit: 0

/home/pcastillo/sen2vm/sen2vm-core/sen2vm-notebook/src/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Sen2VM configuration

In [3]:
# === GENERATE CONFIG.JSON  ===

import os
import json
import re
import shutil
from numpy import double

USERCONF_DIR = os.path.join(WORKDIR, "UserConf")
os.makedirs(USERCONF_DIR, exist_ok=True)

print("UserConf directory:", USERCONF_DIR)

print("Geoid directory:", PATH_GEOID)
# =====================================================
# 1. Extract mission from DATASTRIP
# =====================================================

datastrip_dir = os.path.join(PATH_L1B_DATA, "DATASTRIP")

if not os.path.isdir(datastrip_dir):
    raise RuntimeError(f"DATASTRIP directory not found: {datastrip_dir}")

datastrip_entries = os.listdir(datastrip_dir)
if not datastrip_entries:
    raise RuntimeError(f"No DATASTRIP found in: {datastrip_dir}")

datastrip_name = datastrip_entries[0]

match = re.match(r"(S2[A-C])_OPER_", datastrip_name)
if not match:
    raise RuntimeError("Cannot extract mission (S2A/S2B/S2C) from DATASTRIP name.")

mission = match.group(1)
# =====================================================
# 2. Docker paths inside /workspace
# =====================================================

docker_l1b  = "/data/L1B"
docker_dem  = "/data/DEM"
docker_gipp = f"/data/GIPP/{mission}"
# =====================================================
# 3. Geoid management
# =====================================================

# Detect .gtx inside PATH_GEOID
geoid_files = [f for f in os.listdir(PATH_GEOID) if f.lower().endswith(".gtx")]
if len(geoid_files) == 0:
    raise RuntimeError("No .gtx geoid file found in PATH_GEOID.")

docker_geoid = f"/data/GEOID/{geoid_files[0]}"
# =====================================================
# 4. Locate IERS bulletin on host
# =====================================================

iers_host = None

for f in os.listdir(PATH_IERS):
    if f.startswith("bulletin"):
        iers_host = os.path.join(PATH_IERS, f)
        break

if iers_host is None:
    raise RuntimeError("IERS bulletin not found inside PATH_IERS directory.")

docker_iers = f"/data/IERS/{os.path.basename(iers_host)}"
print(docker_iers)
# =====================================================
# 5. Build config dictionary
# =====================================================

config = {
    "l1b_product": docker_l1b,
    "gipp_folder": docker_gipp,
    "auto_gipp_selection": True,
    "grids_overwriting": True,
    "dem": docker_dem,
    "geoid": docker_geoid,
    "iers": docker_iers,
    "operation": "inverse",
    "deactivate_available_refining": False,
    "steps": {
        "10m_bands": STEPS["10m_bands"],
        "20m_bands": STEPS["20m_bands"],
        "60m_bands": STEPS["60m_bands"]
    },
    "export_alt": True
}

config["inverse_location_additional_info"] = {
    "ul_x": double(LOCATION["ul_x"]),
    "ul_y": double(LOCATION["ul_y"]),
    "lr_x": double(LOCATION["lr_x"]),
    "lr_y": double(LOCATION["lr_y"]),
    "referential": f"EPSG:{UTM_EPSG}",
    "output_folder": "/workspace/output/INVERSE/INVERSE_GRID"
}
# =====================================================
# Save config.json
# =====================================================

config_path = os.path.join(USERCONF_DIR, "config.json")

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Configuration file generated:")
print(config_path)

UserConf directory: /home/pcastillo/sen2vm/WORKDIR/UserConf
Geoid directory: /home/pcastillo/sen2vm/WORKDIR/GEOID
/data/IERS/bulletina-xxxvii-042.txt
Configuration file generated:
/home/pcastillo/sen2vm/WORKDIR/UserConf/config.json


In [4]:
# === GENERATE PARAMS.JSON ===

import os
import json
import re

USERCONF_DIR = os.path.join(WORKDIR, "UserConf")
os.makedirs(USERCONF_DIR, exist_ok=True)

print("UserConf directory:", USERCONF_DIR)

# =====================================================
# Locate GRANULE folders
# =====================================================

GR_TARGET_DIR = os.path.join(PATH_L1B_DATA, "GRANULE")

if not os.path.exists(GR_TARGET_DIR):
    raise RuntimeError("GRANULE directory not found inside L1B SAFE.")

granule_folders = [
    os.path.join(GR_TARGET_DIR, d)
    for d in os.listdir(GR_TARGET_DIR)
    if os.path.isdir(os.path.join(GR_TARGET_DIR, d))
]

print("Found", len(granule_folders), "granule folders.")
# =====================================================
# Extract detectors and bands from JP2
# =====================================================

detectors = set()
bands = set()

pattern = r"_D(\d+)_B(\d{1,2}[A]?)\.jp2$"

for granule in granule_folders:
    img_data_dir = os.path.join(granule, "IMG_DATA")

    if not os.path.isdir(img_data_dir):
        continue

    for fname in os.listdir(img_data_dir):
        match = re.search(pattern, fname)
        if match:
            detectors.add(match.group(1))
            bands.add(f"B{match.group(2)}")

detectors = sorted(detectors)
bands = sorted(bands)

print("Detected detectors:", detectors)
print("Detected bands:", bands)
# =====================================================
# Write params.json
# =====================================================

# Validate keep_detectors: must be a non-empty list (user requirement)
if not isinstance(ORTHO_SETTINGS.get("keep_detectors"), list) or len(ORTHO_SETTINGS["keep_detectors"]) == 0:
    raise RuntimeError("ORTHO_SETTINGS['keep_detectors'] must be a non-empty list of detector ids (e.g. ['01','02']).")

# Filter detectors to only include those selected by the user
keep_detectors_list = ORTHO_SETTINGS["keep_detectors"]
selected_detectors = [d for d in keep_detectors_list if d in detectors]

# Validate that all requested detectors exist
missing_detectors = [d for d in keep_detectors_list if d not in detectors]
if missing_detectors:
    raise RuntimeError(f"Some requested detectors are not available in the product: {missing_detectors}. Available detectors: {detectors}")

# Only include selected detectors and bands in params.json
params = {
    "detectors": sorted(selected_detectors),
    "bands": sorted(ORTHO_SETTINGS["keep_bands"])
}

params_path = os.path.join(USERCONF_DIR, "params.json")

with open(params_path, "w") as f:
    json.dump(params, f, indent=4)

print("params.json written to:", params_path)
print(f"  - Detectors: {params['detectors']}")
print(f"  - Bands: {params['bands']}")

UserConf directory: /home/pcastillo/sen2vm/WORKDIR/UserConf
Found 60 granule folders.
Detected detectors: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Detected bands: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'B8A']
params.json written to: /home/pcastillo/sen2vm/WORKDIR/UserConf/params.json
  - Detectors: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
  - Bands: ['B02']


# Sen2VM run

In [5]:
# === RUN SEN2VM (Docker: BUILD + RUN + CLEAN) ===

import os
import subprocess

# Notebook location (NOT relative to CWD)
notebook_dir = os.getcwd()

dockerfile_dir = os.path.abspath(os.path.join(
    notebook_dir,
    "..", ".."
))

config_inside = "/workspace/UserConf/config.json"
params_inside = "/workspace/UserConf/params.json"

os.makedirs(
    os.path.join(WORKDIR, "output", "INVERSE", "INVERSE_GRID"),
    exist_ok=True
)
# =====================================================
# 1. BUILD DOCKER IMAGE
# =====================================================

print(f"Building Docker image 'sen2vm' from: {dockerfile_dir}")
cmd_build = [
    "docker", "build",
    "-t", "sen2vm",
    dockerfile_dir
]

print("Command:", " ".join(cmd_build), "\n")
subprocess.run(cmd_build, check=True)
print("Docker image built successfully.\n")
# =====================================================
# 2. RUN SEN2VM CONTAINER
# =====================================================

cmd_run = [
    "docker", "run",
    "--rm",
    "-v", f"{PATH_L1B_DATA}:/data/L1B",
    "-v", f"{PATH_DEM}:/data/DEM",
    "-v", f"{PATH_GIPP}:/data/GIPP",
    "-v", f"{PATH_GEOID}:/data/GEOID",
    "-v", f"{PATH_IERS}:/data/IERS",
    "-v", f"{WORKDIR}:/workspace",
    "sen2vm",
    "-c", config_inside,
    "-p", params_inside
]

print("Running Docker container...\n")
print("Command:", " ".join(cmd_run), "\n")
subprocess.run(cmd_run, check=True)
print("\nDocker execution complete.\n")
# =====================================================
# 3. REMOVE DOCKER IMAGE
# =====================================================
if REMOVE_DOCKER_IMAGE:
    print("Removing Docker image 'sen2vm'...")
    subprocess.run(["docker", "rmi", "-f", "sen2vm"], check=True)
    print("Docker images removed.\n")
else:
    print("Docker images kept.\n")

Building Docker image 'sen2vm' from: /home/pcastillo/sen2vm/sen2vm-core
Command: docker build -t sen2vm /home/pcastillo/sen2vm/sen2vm-core 



DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/



Sending build context to Docker daemon  1.718GB
Step 1/5 : FROM ghcr.io/sen2vm/sen2vm-build-env:latest AS launcher
 ---> a5369801b500
Step 2/5 : ENV SEN2VM_VERSION=1.1.4
 ---> Using cache
 ---> 14de44fa18ab
Step 3/5 : WORKDIR /Sen2vm
 ---> Using cache
 ---> b4079a1b81e1
Step 4/5 : RUN curl -L -o sen2vm-core.jar https://github.com/sen2vm/sen2vm-core/releases/download/${SEN2VM_VERSION}/sen2vm-core-${SEN2VM_VERSION}.jar
 ---> Using cache
 ---> 01c51fc5d7d7
Step 5/5 : ENTRYPOINT ["java", "-jar","sen2vm-core.jar"]
 ---> Using cache
 ---> 5be1977daa50
Successfully built 5be1977daa50
Successfully tagged sen2vm:latest
Docker image built successfully.

Running Docker container...

Command: docker run --rm -v /home/pcastillo/sen2vm/WORKDIR/S2B_MSIL1B_20241019T120219_N0511_R023_20241022T154709.SAFE:/data/L1B -v /home/pcastillo/sen2vm/WORKDIR/DEM:/data/DEM -v /home/pcastillo/sen2vm/WORKDIR/GIPP:/data/GIPP -v /home/pcastillo/sen2vm/WORKDIR/GEOID:/data/GEOID -v /home/pcastillo/sen2vm/WORKDIR:/data/I

2026-06-26 09:12:56 [INFO   ] Start Sen2VM 
2026-06-26 09:12:57 [INFO   ] Parsing file /workspace/UserConf/config.json 
2026-06-26 09:12:57 [INFO   ] Reading IERS file at: /data/IERS/bulletina-xxxvii-042.txt 
2026-06-26 09:12:57 [INFO   ] Parsing file /workspace/UserConf/params.json 
2026-06-26 09:12:57 [INFO   ] Detectors list: [DETECTOR_1, DETECTOR_2, DETECTOR_3, DETECTOR_4, DETECTOR_5, DETECTOR_6, DETECTOR_7, DETECTOR_8, DETECTOR_9, DETECTOR_10, DETECTOR_11, DETECTOR_12] 
2026-06-26 09:12:57 [INFO   ] Bands list: [BAND_2] 
2026-06-26 09:12:57 [INFO   ] Find the following datastrip metadata file: /data/L1B/DATASTRIP/S2B_OPER_MSI_L1B_DS_2BPS_20241019T153411_S20241019T120215_N05.11/S2B_OPER_MTD_L1B_DS_2BPS_20241019T153411_S20241019T120215.xml 
2026-06-26 09:12:58 [INFO   ] Initializing: copy of the Orekit-data: /Sen2vm/orekit-data 
2026-06-26 09:12:58 [INFO   ] Orekit-data: /Sen2vm/orekit-data 
2026-06-26 09:13:05 [INFO   ] Get through GIPP folder: /data/GIPP/S2B 
2026-06-26 09:13:05 [


Docker execution complete.

Docker images kept.



    # Generate Orthorectification images

In [6]:
# =====================================================
# INVERSE
# =====================================================
import os
import subprocess
import glob
import json
import re

# =====================================================
# SAFETY CHECKS
# =====================================================
assert os.path.exists(PATH_L1B_DATA), "L1B path missing"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
assert os.path.exists(OUTPUT_FOLDER), "Output folder missing"

# =====================================================
# OUTPUT FOLDERS
# =====================================================
RAW_DIR = os.path.join(OUTPUT_FOLDER, "raw")
OTB_DIR = os.path.join(OUTPUT_FOLDER, "otb_no_georef")
GEOREF_DIR = os.path.join(OUTPUT_FOLDER, "output_georef")
MOSAIC_DIR = os.path.join(OUTPUT_FOLDER, "mosaic")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OTB_DIR, exist_ok=True)
os.makedirs(GEOREF_DIR, exist_ok=True)
os.makedirs(MOSAIC_DIR, exist_ok=True)

# =====================================================
# Locate product name
# =====================================================
product = os.path.basename(os.path.normpath(PATH_L1B_DATA))
print("Product:", product)

# =====================================================
# Locate XML
# =====================================================
xml_list = glob.glob(
    os.path.join(
        PATH_L1B_DATA,
        "S2*_MTD_*.xml"
    )
)

if len(xml_list) == 0:
    raise RuntimeError("No DATASTRIP MTD XML found")

xml_path = xml_list[0]
xml_name = os.path.basename(xml_path)
print("Using XML:", xml_path)

# =====================================================
# Read params.json to get selected bands and detectors
# =====================================================
params_path = os.path.join(WORKDIR, "UserConf", "params.json")
if not os.path.exists(params_path):
    raise RuntimeError(f"params.json not found at {params_path}")

with open(params_path, "r") as f:
    params = json.load(f)

selected_bands = params.get("bands", [])
selected_detectors = params.get("detectors", [])

print(f"Bands from params.json: {selected_bands}")
print(f"Detectors from params.json: {selected_detectors}")

# =====================================================
# Build GDAL Docker
# =====================================================
notebook_dir = os.path.dirname(os.getcwd())
dockerfile_dir = os.path.abspath(os.path.join(
    notebook_dir,
    "src",
    "gdal-latest"
))
print("\n=== BUILDING GDAL CONTAINER ===\n")

cmd_build = [
    "docker", "build",
    "--platform=linux/amd64",
    "-t", "gdal-latest",
    dockerfile_dir
]

print("Command:")
print(" ".join(cmd_build))
subprocess.run(cmd_build, check=True)
print("\nGDAL image built successfully.\n")

# =====================================================
# Locate inverse grids
# =====================================================
INV_FOLDER = os.path.join(OUTPUT_FOLDER, "INVERSE_GRID")
if not os.path.exists(INV_FOLDER):
    raise RuntimeError(f"Inverse folder not found: {INV_FOLDER}")
    
inv_grid_list = glob.glob(
    os.path.join(INV_FOLDER, "*.tif")
)
if len(inv_grid_list) == 0:
    raise RuntimeError("No inverse grids found")
print(f"\nFound {len(inv_grid_list)} inverse grids:\n")

for g in inv_grid_list:
    print("  ", os.path.basename(g))

# =====================================================
# Process all inverse grids
# =====================================================
for inv_grid_path in inv_grid_list:
    test_grid = os.path.basename(inv_grid_path)
    print("\n=================================================")
    print("Processing:", test_grid)
    print("=================================================")

    # =================================================
    # Parse detector / band
    # =================================================
    match = re.search(r"_D(\d+)_B(\d{1,2}[A]?)", test_grid)
    if not match:
        print("Could not parse detector/band")
        continue

    detector = match.group(1)
    band = f"B{match.group(2)}"

    print("Detector:", detector)
    print("Band:", band)

    # =================================================
    # Build subdataset dynamically
    # =================================================
    subdataset = test_grid.replace("INV_L1B", "GEO_L1B")
    subdataset = subdataset.replace(".tif", "")
    print("Subdataset:", subdataset)
    
    # =================================================
    # Determine resolution
    # =================================================
    if band in ["B02", "B03", "B04", "B08"]:
        resolution = 10
    elif band in ["B05", "B06", "B07", "B8A", "B11", "B12"]:
        resolution = 20
    elif band in ["B01", "B09", "B10"]:
        resolution = 60
    else:
        resolution = 10
        
    print("Resolution:", resolution)
    
    # =================================================
    # Compute output size
    # =================================================
    width = int(
        (LOCATION["lr_x"] - LOCATION["ul_x"]) / resolution
    )
    height = int(
        (LOCATION["ul_y"] - LOCATION["lr_y"]) / resolution
    )
    print("Width :", width)
    print("Height:", height)

    # =================================================
    # File paths
    # =================================================
    raw_name = f"raw_D{detector}_{band}.tif"
    ortho_name = f"ortho_D{detector}_{band}.tif"
    georef_name = f"ortho_D{detector}_{band}_georef.tif"
    
    # =================================================
    # Extract raw detector image
    # =================================================
    cmd_extract = [
        "docker", "run",
        "--rm",
        "-v", f"{PATH_L1B_DATA}:/data/L1B",
        "-v", f"{OUTPUT_FOLDER}:/output",
        "gdal-latest",
        "-c",
        f'''
        gdal_translate \
        "SENTINEL2_L1B_WITH_GEOLOC:/data/L1B/{xml_name}:{subdataset}" \
        "/output/raw/{raw_name}"
        '''
    ]
    print("\n=== EXTRACTION ===\n")
    print(" ".join(cmd_extract))

    subprocess.run(cmd_extract, check=True)
    
    # Half-pixel correction (OTB bug fix)
    corrected_ulx = LOCATION["ul_x"] + resolution / 2
    corrected_uly = LOCATION["ul_y"] - resolution / 2

    # =================================================
    # OTB resampling
    # =================================================
    cmd_otb = [
        "docker", "run",
        "--rm",
        "-v", f"{OUTPUT_FOLDER}:/output",
        "orfeotoolbox/otb:9.0.0",
        "otbcli_GridBasedImageResampling",
        "-io.in",
       f"/output/raw/{raw_name}",
        "-io.out",
        f"/output/otb_no_georef/{ortho_name}",
        "-grid.in",
        f"/output/INVERSE_GRID/{test_grid}",
        "-grid.type",
        "loc",
        "-out.ulx",
        str(corrected_ulx),
        "-out.uly",
        str(corrected_uly),
        "-out.spacingx",
        str(resolution),
        "-out.spacingy",
        str(-resolution),
        "-out.sizex",
        str(width),
        "-out.sizey",
        str(height)
    ]

    print("\n=== OTB RESAMPLING ===\n")
    print(" ".join(cmd_otb))

    try:
        subprocess.run(cmd_otb, check=True)
        print("OTB success")
    
    except subprocess.CalledProcessError:
        print("OTB failed (outside grid) → skipping")
        continue

    # ==================a===============================
    # Add georeferencing to correct otb missing one
    # =================================================
    cmd_georef = [
        "docker", "run",
        "--rm",
        "-v", f"{OUTPUT_FOLDER}:/output",
        "gdal-latest",
        "-c",
        f'''
        gdal_translate \
            -a_srs EPSG:{UTM_EPSG} \
            /output/otb_no_georef/{ortho_name} \
            /output/output_georef/{georef_name}
        '''
    ]

    print("\n=== ADD GEOREF ===\n")
    print(" ".join(cmd_georef))
    subprocess.run(cmd_georef, check=True)
    print("\nFinished:", georef_name)

# =====================================================
# Mosaic
# =====================================================
print("\n=== MOSAIC CREATION ===\n")
bands = set()
for f in glob.glob(os.path.join(GEOREF_DIR, "*.tif")):
    match = re.search(r"_B(\d{2}[A]?)", f)
    if match:
        bands.add(f"B{match.group(1)}")
print("Bands detected:", bands)

for band in bands:

    print("\n----------------------------------------")
    print("Creating mosaic for band:", band)
    print("----------------------------------------")
    # fichiers correspondant à la bande
    input_files = glob.glob(
        os.path.join(GEOREF_DIR, f"*_{band}_georef.tif")
    )
    if len(input_files) == 0:
        print("No ortho images found for band", band)
        continue
    input_files = sorted(input_files)
    
    output_path = os.path.join(
        MOSAIC_DIR,
        f"ORTHO_mosaic_{band}.tif"
    )
    print("Found", len(input_files), "files")
    for f in input_files:
        print("  ", os.path.basename(f))
    
    gdal_cmd = "gdal_merge.py " \
        f"-o /output/mosaic/ORTHO_mosaic_{band}.tif " \
        "-of GTiff " \
        "-co COMPRESS=LZW " \
        "-co TILED=YES " \
        "-ot UInt16 " \
        "-n 0 -a_nodata 0 "

    for f in input_files:
        fname = os.path.basename(f)
        gdal_cmd += f"/output/output_georef/{fname} "

    cmd_mosaic = [
        "docker", "run",
        "--rm",
        "-v", f"{OUTPUT_FOLDER}:/output",
        "gdal-latest",
        "-c",
        gdal_cmd
    ]

    print("\nRunning mosaic:\n", gdal_cmd)
    subprocess.run(cmd_mosaic, check=True)
    print("Mosaic written →", output_path)

# =====================================================
# Cleanup docker image
# =====================================================
if REMOVE_DOCKER_IMAGE:
    print("Removing gdal-latest image...\n")
    subprocess.run(["docker", "rmi", "-f", "gdal-latest"], check=True)
    print("GDAL image removed.\n")
else:
    print("Docker images kept (faster next run, ~4 GB disk usage).\n")

Product: S2B_MSIL1B_20241019T120219_N0511_R023_20241022T154709.SAFE
Using XML: /home/pcastillo/sen2vm/WORKDIR/S2B_MSIL1B_20241019T120219_N0511_R023_20241022T154709.SAFE/S2B_OPER_MTD_SAFL1B_PDMC_20241022T154709_R023_V20241019T120217_20241019T120235.xml
Bands from params.json: ['B02']
Detectors from params.json: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

=== BUILDING GDAL CONTAINER ===

Command:
docker build --platform=linux/amd64 -t gdal-latest /home/pcastillo/sen2vm/sen2vm-core/sen2vm-notebook/src/gdal-latest
Sending build context to Docker daemon  3.072kB
Step 1/10 : FROM ubuntu:22.04
 ---> 962f6cadeae0
Step 2/10 : ENV DEBIAN_FRONTEND=noninteractive
 ---> Using cache
 ---> e6e9ea6fb1e0
Step 3/10 : RUN apt-get update && apt-get install -y     build-essential     git     cmake     pkg-config     curl     wget     zip     libcurl4-openssl-dev     libproj-dev     proj-bin     libgeos-dev     libsqlite3-dev     libtiff-dev     libjpeg-dev     libpng-dev     l

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/



Input file size is 2552, 11520
0...10...20...30...40...50...60...70...80...90...100 - done.

=== OTB RESAMPLING ===

docker run --rm -v /home/pcastillo/sen2vm/WORKDIR/output/INVERSE:/output orfeotoolbox/otb:9.0.0 otbcli_GridBasedImageResampling -io.in /output/raw/raw_D12_B02.tif -io.out /output/otb_no_georef/ortho_D12_B02.tif -grid.in /output/INVERSE_GRID/S2B_OPER_INV_L1B_DS_2BPS_20241019T153411_S20241019T120215_D12_B02.tif -grid.type loc -out.ulx 283915.0 -out.uly 3641655.0 -out.spacingx 10 -out.spacingy -10 -out.sizex 7037 -out.sizey 3334
2026-06-26 09:16:25 (INFO) GridBasedImageResampling: Default RAM limit for OTB is 256 MB
2026-06-26 09:16:25 (INFO) GridBasedImageResampling: GDAL maximum cache size is 799 MB
2026-06-26 09:16:25 (INFO) GridBasedImageResampling: OTB will use at most 4 threads
2026-06-26 09:16:25 (INFO): Loading metadata from official product
2026-06-26 09:16:25 (INFO): Loading metadata from official product
2026-06-26 09:16:25 (INFO) GridBasedImageResampling: Grid i